# Runnable Passthrough Utilities

The `passthrough.py` module defines identity helpers and serializable Runnables for preserving inputs, adding calculated dictionary fields, and selecting dictionary keys.

`RunnablePassthrough` returns its input unchanged and may run an optional side-effect function. `RunnableAssign` combines an input dictionary with fields generated by a parallel mapper. `RunnablePick` extracts one or more keys from dictionary input.

## Functions

1. `identity`: Returns the supplied input unchanged.
   * **Syntax:**
     ```python
     identity(
         x: Other # Input value
     ) -> Other
     ```

2. `aidentity`: Asynchronously returns the supplied input unchanged.
   * **Syntax:**
     ```python
     async aidentity(
         x: Other # Input value
     ) -> Other
     ```

# RunnablePassthrough

`RunnablePassthrough` is a serializable Runnable that returns its input unchanged.

An optional synchronous or asynchronous function may be executed for side effects without changing the returned value. For streamed input, the configured function receives the aggregated input after the chunks have been passed through.

## Bases

- `RunnableSerializable[Other, Other]`

## Attributes

1. `input_type`: Stores an optional explicit type for both the input and output.
   * **Type:**
     ```python
     input_type: type[Other] | None = None
     ```

2. `func`: Stores an optional synchronous side-effect function.

   The function may accept only the input or both the input and its `RunnableConfig`.

   * **Type:**
     ```python
     func: Callable[
         [Other],
         None
     ]
     | Callable[
         [Other, RunnableConfig],
         None
     ]
     | None = None
     ```

3. `afunc`: Stores an optional asynchronous side-effect function.

   The function may accept only the input or both the input and its `RunnableConfig`.

   * **Type:**
     ```python
     afunc: Callable[
         [Other],
         Awaitable[None]
     ]
     | Callable[
         [Other, RunnableConfig],
         Awaitable[None]
     ]
     | None = None
     ```

### Properties

1. `InputType`: Returns the configured input type or `Any` when no explicit type is supplied.
   * **Type:**
     ```python
     InputType: Any
     ```

2. `OutputType`: Returns the configured input type or `Any` when no explicit type is supplied.
   * **Type:**
     ```python
     OutputType: Any
     ```

### Methods

1. `__init__`: Creates a passthrough Runnable.

   When `func` is an asynchronous function, it is stored as `afunc` automatically.

   * **Syntax:**
     ```python
     __init__(
         self,
         func: Callable[
             [Other],
             None
         ]
         | Callable[
             [Other, RunnableConfig],
             None
         ]
         | Callable[
             [Other],
             Awaitable[None]
         ]
         | Callable[
             [Other, RunnableConfig],
             Awaitable[None]
         ]
         | None = None, # Optional synchronous or asynchronous side-effect function
         afunc: Callable[
             [Other],
             Awaitable[None]
         ]
         | Callable[
             [Other, RunnableConfig],
             Awaitable[None]
         ]
         | None = None, # Optional asynchronous side-effect function
         *,
         input_type: type[Other] | None = None, # Explicit input and output type
         **kwargs: Any # Additional Runnable fields
     ) -> None
     ```

2. `is_lc_serializable`: Indicates that the passthrough Runnable supports LangChain serialization.
   * **Syntax:**
     ```python
     @classmethod
     is_lc_serializable(
         cls
     ) -> bool
     ```

3. `get_lc_namespace`: Returns the LangChain serialization namespace.
   * **Syntax:**
     ```python
     @classmethod
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

4. `assign`: Creates a `RunnableAssign` that merges dictionary input with calculated fields.

   Each keyword value may be a Runnable, callable, or mapping that can be converted into a `RunnableParallel`.

   * **Syntax:**
     ```python
     @classmethod
     assign(
         cls,
         **kwargs: Runnable[
             dict[str, Any],
             Any
         ]
         | Callable[
             [dict[str, Any]],
             Any
         ]
         | Mapping[
             str,
             Runnable[
                 dict[str, Any],
                 Any
             ]
             | Callable[
                 [dict[str, Any]],
                 Any
             ]
         ] # Fields to calculate and merge into the input dictionary
     ) -> RunnableAssign
     ```

5. `invoke`: Runs the optional synchronous side-effect function and returns the original input unchanged.

   * **Syntax:**
     ```python
     invoke(
         self,
         input: Other, # Input value to pass through
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> Other
     ```

6. `ainvoke`: Runs the optional asynchronous or synchronous side-effect function and asynchronously returns the original input unchanged.

   The asynchronous function is preferred when both functions are configured.

   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: Other, # Input value to pass through
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any | None # Additional invocation arguments
     ) -> Other
     ```

7. `transform`: Passes through an iterator of input chunks.

   When a synchronous side-effect function is configured, the chunks are combined using addition when supported. If chunks cannot be added, the latest chunk becomes the final value. The function is called once after the stream ends.

   * **Syntax:**
     ```python
     transform(
         self,
         input: Iterator[Other], # Input chunks to pass through
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional transformation arguments
     ) -> Iterator[Other]
     ```

8. `atransform`: Asynchronously passes through an input stream.

   When a side-effect function is configured, the chunks are aggregated when possible and the asynchronous or synchronous function is called once after the stream ends.

   * **Syntax:**
     ```python
     async atransform(
         self,
         input: AsyncIterator[Other], # Asynchronous input chunks to pass through
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional transformation arguments
     ) -> AsyncIterator[Other]
     ```

9. `stream`: Passes one input through the synchronous streaming interface.
   * **Syntax:**
     ```python
     stream(
         self,
         input: Other, # Input value to pass through
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional streaming arguments
     ) -> Iterator[Other]
     ```

10. `astream`: Passes one input through the asynchronous streaming interface.
    * **Syntax:**
      ```python
      async astream(
          self,
          input: Other, # Input value to pass through
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any # Additional streaming arguments
      ) -> AsyncIterator[Other]
      ```

# RunnableAssign

`RunnableAssign` is a serializable Runnable that calculates new values from dictionary input and merges them with the original dictionary.

The calculated values are produced by a `RunnableParallel`. When a generated key is also present in the original dictionary, the generated value replaces the original value.

A `ValueError` is raised when the input is not a dictionary.

## Bases

- `RunnableSerializable[dict[str, Any], dict[str, Any]]`

## Attributes

1. `mapper`: Stores the parallel Runnable whose output fields are merged into the input dictionary.
   * **Type:**
     ```python
     mapper: RunnableParallel
     ```

### Properties

1. `config_specs`: Returns the configurable-field specifications exposed by the mapper.
   * **Type:**
     ```python
     config_specs: list[
         ConfigurableFieldSpec
     ]
     ```

### Methods

1. `__init__`: Creates an assignment Runnable from a parallel mapper.
   * **Syntax:**
     ```python
     __init__(
         self,
         mapper: RunnableParallel[
             dict[str, Any]
         ], # Parallel mapper used to calculate new fields
         **kwargs: Any # Additional Runnable fields
     ) -> None
     ```

2. `is_lc_serializable`: Indicates that the assignment Runnable supports LangChain serialization.
   * **Syntax:**
     ```python
     @classmethod
     is_lc_serializable(
         cls
     ) -> bool
     ```

3. `get_lc_namespace`: Returns the LangChain serialization namespace.
   * **Syntax:**
     ```python
     @classmethod
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

4. `get_name`: Returns the Runnable's display name.

   When no explicit name is supplied, the generated name contains the keys defined by the mapper.

   * **Syntax:**
     ```python
     get_name(
         self,
         suffix: str | None = None, # Optional name suffix
         *,
         name: str | None = None # Optional replacement name
     ) -> str
     ```

5. `get_input_schema`: Returns the mapper's dictionary input schema when available.

   When the mapper exposes only a root schema, schema generation falls back to the base Runnable implementation.

   * **Syntax:**
     ```python
     get_input_schema(
         self,
         config: RunnableConfig | None = None # Runtime configuration used for schema generation
     ) -> TypeBaseModel
     ```

6. `get_output_schema`: Returns a schema representing the merged dictionary output.

   When both the mapper's input and output are dictionary models, their fields are combined into a model named `RunnableAssignOutput`. When only the mapper output is a dictionary model, that output schema is returned directly.

   * **Syntax:**
     ```python
     get_output_schema(
         self,
         config: RunnableConfig | None = None # Runtime configuration used for schema generation
     ) -> TypeBaseModel
     ```

7. `get_graph`: Returns the mapper's graph with an additional passthrough route between its first and last nodes.

   The passthrough route represents the original dictionary values that are preserved alongside the mapper output.

   * **Syntax:**
     ```python
     get_graph(
         self,
         config: RunnableConfig | None = None # Runtime configuration used to build the graph
     ) -> Graph
     ```

8. `invoke`: Synchronously calculates the mapped fields and merges them into the input dictionary.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: dict[str, Any], # Dictionary to extend
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> dict[str, Any]
     ```

9. `ainvoke`: Asynchronously calculates the mapped fields and merges them into the input dictionary.
   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: dict[str, Any], # Dictionary to extend
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> dict[str, Any]
     ```

10. `transform`: Streams preserved input fields and mapped output fields.

    Input chunks are duplicated into passthrough and mapper streams. Keys generated by the mapper are removed from passthrough chunks so that mapper values can replace them. The mapper stream is started through the configured executor.

    * **Syntax:**
      ```python
      transform(
          self,
          input: Iterator[
              dict[str, Any]
          ], # Input dictionary chunks
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any | None # Additional transformation arguments
      ) -> Iterator[
          dict[str, Any]
      ]
      ```

11. `atransform`: Asynchronously streams preserved input fields and mapped output fields.

    The input stream is duplicated asynchronously, and the first mapper output is started as a task while passthrough chunks are consumed.

    * **Syntax:**
      ```python
      async atransform(
          self,
          input: AsyncIterator[
              dict[str, Any]
          ], # Asynchronous input dictionary chunks
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any # Additional transformation arguments
      ) -> AsyncIterator[
          dict[str, Any]
      ]
      ```

12. `stream`: Processes one dictionary through the synchronous streaming interface.
    * **Syntax:**
      ```python
      stream(
          self,
          input: dict[str, Any], # Dictionary to extend
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any # Additional streaming arguments
      ) -> Iterator[
          dict[str, Any]
      ]
      ```

13. `astream`: Processes one dictionary through the asynchronous streaming interface.
    * **Syntax:**
      ```python
      async astream(
          self,
          input: dict[str, Any], # Dictionary to extend
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any # Additional streaming arguments
      ) -> AsyncIterator[
          dict[str, Any]
      ]
      ```

# RunnablePick

`RunnablePick` is a serializable Runnable that extracts selected keys from dictionary input.

When `keys` is a string, it returns the value associated with that key or `None` when the key is missing. When `keys` is a list, it returns an `AddableDict` containing the selected keys that exist, or `None` when none of them exist.

A `ValueError` is raised when the input is not a dictionary.

## Bases

- `RunnableSerializable[dict[str, Any], Any]`

## Attributes

1. `keys`: Stores one key or a list of keys to extract from each input dictionary.
   * **Type:**
     ```python
     keys: str | list[str]
     ```

### Methods

1. `__init__`: Creates a key-selection Runnable.
   * **Syntax:**
     ```python
     __init__(
         self,
         keys: str | list[str], # Key or keys to select
         **kwargs: Any # Additional Runnable fields
     ) -> None
     ```

2. `is_lc_serializable`: Indicates that the key-selection Runnable supports LangChain serialization.
   * **Syntax:**
     ```python
     @classmethod
     is_lc_serializable(
         cls
     ) -> bool
     ```

3. `get_lc_namespace`: Returns the LangChain serialization namespace.
   * **Syntax:**
     ```python
     @classmethod
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

4. `get_name`: Returns the Runnable's display name.

   When no explicit name is supplied, the generated name contains the selected key names.

   * **Syntax:**
     ```python
     get_name(
         self,
         suffix: str | None = None, # Optional name suffix
         *,
         name: str | None = None # Optional replacement name
     ) -> str
     ```

5. `invoke`: Synchronously extracts the configured key or keys from one dictionary.
   * **Syntax:**
     ```python
     invoke(
         self,
         input: dict[str, Any], # Dictionary from which values are selected
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> Any
     ```

6. `ainvoke`: Asynchronously extracts the configured key or keys from one dictionary.
   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: dict[str, Any], # Dictionary from which values are selected
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional invocation arguments
     ) -> Any
     ```

7. `transform`: Extracts the configured key or keys from each synchronous input chunk.

   Chunks whose selected result is `None` are not yielded.

   * **Syntax:**
     ```python
     transform(
         self,
         input: Iterator[
             dict[str, Any]
         ], # Input dictionary chunks
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional transformation arguments
     ) -> Iterator[Any]
     ```

8. `atransform`: Extracts the configured key or keys from each asynchronous input chunk.

   Chunks whose selected result is `None` are not yielded.

   * **Syntax:**
     ```python
     async atransform(
         self,
         input: AsyncIterator[
             dict[str, Any]
         ], # Asynchronous input dictionary chunks
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional transformation arguments
     ) -> AsyncIterator[Any]
     ```

9. `stream`: Selects values from one dictionary through the synchronous streaming interface.
   * **Syntax:**
     ```python
     stream(
         self,
         input: dict[str, Any], # Dictionary from which values are selected
         config: RunnableConfig | None = None, # Runtime configuration
         **kwargs: Any # Additional streaming arguments
     ) -> Iterator[Any]
     ```

10. `astream`: Selects values from one dictionary through the asynchronous streaming interface.
    * **Syntax:**
      ```python
      async astream(
          self,
          input: dict[str, Any], # Dictionary from which values are selected
          config: RunnableConfig | None = None, # Runtime configuration
          **kwargs: Any # Additional streaming arguments
      ) -> AsyncIterator[Any]
      ```